# Inline Decision Budgets for E-Commerce Web Attack Detection

Experiment notebook 1 of 1 for the ICACS paper. Runs entirely on free Colab CPU.

## How to use this notebook (read this first)

1. Open this file in Google Colab: go to https://colab.research.google.com, click **File > Upload notebook**, and pick this `.ipynb`.
2. Make sure you are on a CPU runtime. Click **Runtime > Change runtime type**, set Hardware accelerator to **None**, click Save. Do not use a GPU. The paper measures CPU inference cost, so a GPU runtime would make the timings meaningless.
3. Run the cells **in order, top to bottom**. Click a cell, then press `Shift + Enter`. Wait for the `[*]` next to a cell to turn into a number before moving to the next one.
4. The first run must be a **smoke test**. In Cell 2 the setting `SMOKE = True` is already set. Leave it. The smoke test uses a small slice of data and small models and finishes in roughly 2 to 4 minutes. Its job is to prove every cell works, not to produce paper numbers.
5. When the smoke test finishes with no errors, go back to Cell 2, change `SMOKE = True` to `SMOKE = False`, then click **Runtime > Restart session**, then **Runtime > Run all**. That is the real run. Expect roughly 20 to 40 minutes.
6. Everything is saved to your Google Drive as it goes. If Colab disconnects mid-run, just re-run all cells: finished tiers are skipped automatically and the run resumes.

## What gets saved to Drive

Everything lands in `MyDrive/ICACS_inline/`:

```
ICACS_inline/
  data/                 downloaded CSIC files (downloaded once, reused forever)
  cache/                parsed requests and extracted features (so you never re-parse)
  runs/smoke/           results of smoke runs
  runs/full/            results of the real run
    results.csv         one row per tier, the main results table
    history/            per-boosting-round metrics for each GBDT tier
    timing/             per-request latency measurements
    figures/            Pareto plots as PNG
    manifest.json       versions, settings, seeds, timestamps
```

## What the experiment does

Four to six detector tiers, from a logistic regression up to a char n-gram TF-IDF model, all deciding on a **single HTTP request** with no session history. For each tier we record detection quality at **matched false-positive rates**, single-request latency at p50 and p99, and a hardware-independent compute cost (tree node visits). The question is which tier you should actually deploy when the detector has to answer before the page is served.

---
## Cell 1. Connect Google Drive

When you run this cell a link and a box appear. Click the link, pick your Google account, click **Continue**, copy the code it gives you, paste it into the box, and press Enter. If Colab instead shows a popup asking for permission, click **Connect to Google Drive** and then **Allow**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/ICACS_inline'
for sub in ['data', 'cache', 'runs']:
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
print('Drive is connected. Project folder:', ROOT)
print('Contents so far:', os.listdir(ROOT))

---
## Cell 2. Settings

**This is the only cell you edit.**

Leave `SMOKE = True` for your first run. After the smoke test passes, change it to `False` and do a full restart and run.

In [ ]:
SMOKE = True          # <<<<<< change to False for the real run, then Runtime > Restart session > Run all

SEED = 42             # random seed, keep fixed so results are reproducible
TIMING_SAMPLE = 300 if SMOKE else 3000     # how many requests to time individually
TIMING_REPEATS = 3 if SMOKE else 7         # repeats per request, we keep the fastest
TIMING_SWEEPS = 2 if SMOKE else 5          # full passes over all tiers, tier order reshuffled each pass
N_BOOTSTRAP  = 200 if SMOKE else 1000      # bootstrap resamples for recall confidence intervals
SMOKE_ROWS = 6000                          # rows used in smoke mode
FPR_TARGETS = [0.001, 0.01]                # matched false-positive rates to compare tiers at
REALISTIC_ATTACK_RATE = 0.01               # attacks as 1% of traffic, closer to a real storefront
DEADLINES_US = [250, 500, 1000, 2000, 5000] # decision deadlines in microseconds, for the budget curve

RUN_KIND = 'smoke' if SMOKE else 'full'
print('Mode:', RUN_KIND.upper())
print('If this says SMOKE and you wanted the real run, edit SMOKE above and re-run this cell.')

---
## Cell 3. Install and import libraries

This pins versions so your results stay reproducible. It may print a message about restarting the runtime; you can ignore that, nothing here needs a restart.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightgbm==4.5.0', 'scikit-learn==1.5.2', 'pandas'], check=False)

# Force single-threaded execution. This is essential: the paper measures the cost of one
# request on one core, and multi-threading would make small and large models look identical.
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[v] = '1'

import re, math, json, time, warnings, platform, urllib.parse, urllib.request
import numpy as np, pandas as pd
import lightgbm as lgb, sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

RUN_DIR = os.path.join(ROOT, 'runs', RUN_KIND)
for sub in ['history', 'timing', 'figures']:
    os.makedirs(os.path.join(RUN_DIR, sub), exist_ok=True)

print('lightgbm', lgb.__version__, '| sklearn', sklearn.__version__, '| numpy', np.__version__)
print('cpu:', platform.processor() or 'unknown', '|', os.cpu_count(), 'cores visible')
print('results will be written to:', RUN_DIR)

---
## Cell 4. Download the CSIC 2010 dataset

Downloads three plain text files (about 56 MB total) from a public GitHub mirror. No Kaggle account needed.

They are saved to Drive, so this only ever downloads once. On later runs it prints "already present" and moves on.

The cell then checks the file sizes against known values. If a download was corrupted you will see a warning and should just re-run the cell.

In [ ]:
DATA_DIR = os.path.join(ROOT, 'data')
BASE = 'https://raw.githubusercontent.com/msudol/Web-Application-Attack-Datasets/master/OriginalDataSets/csic_2010'
FILES = {
    'normalTrafficTraining.txt': 20148988,
    'normalTrafficTest.txt':     20151204,
    'anomalousTrafficTest.txt':  15734523,
}

for fname, expected in FILES.items():
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest) and os.path.getsize(dest) == expected:
        print(f'{fname}: already present ({expected:,} bytes)')
        continue
    print(f'{fname}: downloading ...')
    urllib.request.urlretrieve(f'{BASE}/{fname}', dest)
    got = os.path.getsize(dest)
    flag = 'OK' if got == expected else 'SIZE MISMATCH - re-run this cell'
    print(f'{fname}: {got:,} bytes  [{flag}]')

print('\nDone. Files in', DATA_DIR, '->', sorted(os.listdir(DATA_DIR)))

---
## Cell 5. Parse the raw HTTP requests

The files are raw HTTP requests separated by blank lines. This turns them into one row per request with the method, URL, and POST body.

Expected counts, which the cell checks for you: **36,000 normal** and **25,065 attack** requests, 61,065 total. If you see different numbers, something went wrong with the download.

The parsed table is cached to Drive as a Parquet file. Later runs load the cache in a second instead of re-parsing.

In [ ]:
CACHE = os.path.join(ROOT, 'cache')
parsed_path = os.path.join(CACHE, 'csic_parsed.pkl')

REQ_LINE = re.compile(r'^(GET|POST|PUT|DELETE|HEAD)\s+(\S+)\s+HTTP/1\.[01]$')

def parse_file(path, label):
    recs = []
    with open(path, 'r', encoding='latin-1') as f:
        lines = f.read().split('\n')
    i, n = 0, len(lines)
    while i < n:
        m = REQ_LINE.match(lines[i].strip())
        if not m:
            i += 1
            continue
        method, url = m.group(1), m.group(2)
        headers, j = {}, i + 1
        while j < n and lines[j].strip() != '':
            if ':' in lines[j]:
                k, v = lines[j].split(':', 1)
                headers[k.strip().lower()] = v.strip()
            j += 1
        body = ''
        if int(headers.get('content-length', 0) or 0) > 0:
            k = j + 1
            while k < n and lines[k].strip() == '':
                k += 1
            if k < n:
                body, j = lines[k], k
        recs.append({'method': method, 'url': url, 'body': body,
                     'n_headers': len(headers), 'label': label})
        i = j + 1
    return recs

if os.path.exists(parsed_path):
    df = pd.read_pickle(parsed_path)
    print('loaded parsed cache:', parsed_path)
else:
    t0 = time.time()
    recs = (parse_file(os.path.join(DATA_DIR, 'normalTrafficTraining.txt'), 0)
            + parse_file(os.path.join(DATA_DIR, 'anomalousTrafficTest.txt'), 1))
    df = pd.DataFrame(recs)
    df.to_pickle(parsed_path)
    print(f'parsed in {time.time()-t0:.1f}s and cached to Drive')

n_norm, n_atk = int((df.label == 0).sum()), int((df.label == 1).sum())
print(f'normal requests: {n_norm:,}   (expected 36,000)')
print(f'attack requests: {n_atk:,}   (expected 25,065)')
print(f'total:           {len(df):,}')
assert n_norm == 36000 and n_atk == 25065, 'Unexpected counts. Delete the cache file and re-run Cells 4 and 5.'
print('\nmethods:', df.method.value_counts().to_dict())
df.head(3)

---
## Cell 6. Feature extraction

Two feature sets, deliberately separated because their **cost** is part of what we are measuring:

- **CHEAP**: things you can read straight off the request line. Lengths, counts of special characters, entropy, parameter counts. No decoding, no pattern matching.
- **EXPENSIVE**: URL-decoding (twice, to catch double encoding), then keyword and regex matching for SQL injection, XSS and path traversal.

A real inline filter pays for whichever set it uses, on every single request. The paper asks whether the expensive set earns its cost.

Note for the write-up: in CSIC 2010 every request carries identical headers, so all signal lives in the URL, query string and POST body. Header-based features would be useless here, and this is worth stating as a limitation.

In [ ]:
SPECIALS = "%'\"<>;()=&/\\.-+ "
SQL_KW  = ['select','union','drop','insert','update','delete','from','where','or 1','--',';--','exec','waitfor']
XSS_KW  = ['<script','javascript:','onerror','onload','alert(','<img','<iframe','document.cookie']
TRAV_KW = ['../','..\\','/etc/','c:\\','%2e%2e','/bin/']

def entropy(s):
    if not s: return 0.0
    counts = {}
    for ch in s: counts[ch] = counts.get(ch, 0) + 1
    n = len(s)
    return -sum((v/n) * math.log2(v/n) for v in counts.values())

def feats_cheap(r):
    url, body = r['url'], r['body']
    path, q = (url.split('?', 1) + [''])[:2]
    raw = q + '&' + body
    d = {'is_post': float(r['method'] == 'POST'), 'is_put': float(r['method'] == 'PUT'),
         'len_path': len(path), 'len_query': len(q), 'len_body': len(body),
         'n_params': raw.count('='), 'n_amp': raw.count('&'),
         'n_headers': r['n_headers'], 'has_body': float(bool(body)),
         'depth_path': path.count('/'), 'ent_raw': entropy(raw),
         'digit_ratio': sum(c.isdigit() for c in raw) / (len(raw) + 1),
         'alpha_ratio': sum(c.isalpha() for c in raw) / (len(raw) + 1),
         'nonascii': sum(ord(c) > 127 for c in raw),
         'max_param_len': max([len(p) for p in raw.split('&')] + [0])}
    for ch in SPECIALS:
        d['c_' + str(ord(ch))] = raw.count(ch)
    return d

def feats_expensive(r):
    url, body = r['url'], r['body']
    q = url.split('?', 1)[1] if '?' in url else ''
    raw = q + '&' + body
    dec1 = urllib.parse.unquote_plus(raw)
    dec2 = urllib.parse.unquote_plus(dec1)
    low = dec2.lower()
    return {'dec_gain': len(raw) - len(dec1), 'double_dec_gain': len(dec1) - len(dec2),
            'n_sql': sum(low.count(k) for k in SQL_KW),
            'n_xss': sum(low.count(k) for k in XSS_KW),
            'n_trav': sum(low.count(k) for k in TRAV_KW),
            'has_quote_dec': float("'" in dec2 or '"' in dec2),
            'has_angle_dec': float('<' in dec2 or '>' in dec2),
            'has_null': float('\x00' in dec2), 'ent_dec': entropy(dec2),
            'n_space_dec': dec2.count(' '),
            'rx_sqli': float(bool(re.search(r"(\bor\b|\band\b)\s+\d+\s*=\s*\d+", low))),
            'rx_cmd': float(bool(re.search(r"[;|`]\s*(cat|ls|wget|curl|nc)\b", low)))}

def payload_text(r):
    q = r['url'].split('?', 1)[1] if '?' in r['url'] else ''
    return urllib.parse.unquote_plus(q) + ' ' + urllib.parse.unquote_plus(r['body'])

records = df.to_dict('records')
feat_path = os.path.join(CACHE, 'csic_features.pkl')

if os.path.exists(feat_path):
    F = pd.read_pickle(feat_path)
    print('loaded feature cache')
    cost_cheap_us = cost_exp_us = float('nan')
else:
    t0 = time.time(); Xc = pd.DataFrame([feats_cheap(r) for r in records]); cost_cheap_us = 1e6*(time.time()-t0)/len(records)
    t0 = time.time(); Xe = pd.DataFrame([feats_expensive(r) for r in records]); cost_exp_us = 1e6*(time.time()-t0)/len(records)
    F = pd.concat([Xc, Xe], axis=1)
    F.to_pickle(feat_path)
    print(f'bulk extraction cost: CHEAP {cost_cheap_us:.1f} us/request, EXPENSIVE +{cost_exp_us:.1f} us/request')

CHEAP_COLS = [c for c in F.columns if c not in
              ['dec_gain','double_dec_gain','n_sql','n_xss','n_trav','has_quote_dec',
               'has_angle_dec','has_null','ent_dec','n_space_dec','rx_sqli','rx_cmd']]
FULL_COLS = list(F.columns)
TEXTS = [payload_text(r) for r in records]
y_all = df.label.values

print(f'features: {len(CHEAP_COLS)} cheap, {len(FULL_COLS)} full, on {len(F):,} requests')

---
## Cell 7. Train / validation / test split

A three-way split. Validation is used only to pick the score threshold that hits each target false-positive rate; test is used only for the numbers that go in the paper. Never tune on test.

In smoke mode the data is cut down to a few thousand rows so everything runs fast.

In [ ]:
idx = np.arange(len(F))
if SMOKE:
    rng = np.random.RandomState(SEED)
    idx = rng.choice(idx, size=min(SMOKE_ROWS, len(idx)), replace=False)
    print(f'SMOKE MODE: using {len(idx):,} of {len(F):,} requests')

y = y_all[idx]
i_tr, i_tmp = train_test_split(idx, test_size=0.4, random_state=SEED, stratify=y)
i_va, i_te = train_test_split(i_tmp, test_size=0.5, random_state=SEED, stratify=y_all[i_tmp])

print(f'train {len(i_tr):,}  |  validation {len(i_va):,}  |  test {len(i_te):,}')
print(f'attack share: train {y_all[i_tr].mean():.3f}, val {y_all[i_va].mean():.3f}, test {y_all[i_te].mean():.3f}')

np.savez(os.path.join(RUN_DIR, 'splits.npz'), train=i_tr, val=i_va, test=i_te)
print('splits saved to Drive')

---
## Cell 8. Define the detector tiers

Six tiers spanning the cost range, from a linear model to a char n-gram text model. Each is a plausible thing someone would actually deploy.

`featset` says which feature set the tier pays for. `T6_tfidf` bypasses the tabular features entirely and vectorizes the decoded payload, which is what most published CSIC results do.

In [ ]:
def tier_defs(smoke):
    s = (lambda big, small: small if smoke else big)
    return [
        dict(name='T1_logreg',       kind='lr',    featset='cheap', params={}),
        dict(name='T2_tree_d4',      kind='dt',    featset='cheap', params=dict(max_depth=4)),
        dict(name='T3_lgbm_50x8',    kind='lgb',   featset='cheap',
             params=dict(n_estimators=s(50, 20),  num_leaves=8,   learning_rate=0.2)),
        dict(name='T4_lgbm_200x31',  kind='lgb',   featset='cheap',
             params=dict(n_estimators=s(200, 60), num_leaves=31,  learning_rate=0.1)),
        dict(name='T5_lgbm_800x127', kind='lgb',   featset='cheap',
             params=dict(n_estimators=s(800, 100), num_leaves=127, learning_rate=0.05)),
        dict(name='T4f_lgbm_200x31_full', kind='lgb', featset='full',
             params=dict(n_estimators=s(200, 60), num_leaves=31,  learning_rate=0.1)),
        dict(name='T6_tfidf_lr',     kind='tfidf', featset='text',
             params=dict(max_features=s(20000, 3000), ngram_range=(2, 4), min_df=5)),
    ]

TIERS = tier_defs(SMOKE)
print(f'{len(TIERS)} tiers defined:')
for t in TIERS:
    print(f"  {t['name']:24s} kind={t['kind']:6s} features={t['featset']}")

---
## Cell 9. Train every tier and record per-round history

For the gradient boosting tiers, the validation AUC and logloss are recorded **after every boosting round** and written to `history/<tier>.csv`. That is the closest thing this setup has to epochs, and it lets you plot where each model stops improving.

Results are appended to `results.csv` **as each tier finishes**, so a disconnect never costs you more than one tier. Re-running the cell skips tiers that are already in the file.

In [ ]:
RESULTS_CSV = os.path.join(RUN_DIR, 'results.csv')

def load_done():
    if os.path.exists(RESULTS_CSV):
        d = pd.read_csv(RESULTS_CSV)
        return d, set(d['tier'])
    return pd.DataFrame(), set()

def node_visits_lgb(model, X):
    """Exact average number of decision nodes traversed per prediction."""
    leaf = model.predict(X, pred_leaf=True)
    leaf = np.atleast_2d(leaf)
    dump = model.booster_.dump_model() if hasattr(model, 'booster_') else model.dump_model()
    depth_maps = []
    for tree in dump['tree_info']:
        dm = {}
        stack = [(tree['tree_structure'], 0)]
        while stack:
            nd, d = stack.pop()
            if 'leaf_index' in nd:
                dm[nd['leaf_index']] = d
            else:
                stack.append((nd['left_child'], d + 1))
                stack.append((nd['right_child'], d + 1))
        depth_maps.append(dm)
    step = max(1, len(leaf) // 500)
    tot = [sum(depth_maps[t].get(int(leaf[i, t]), 0) for t in range(leaf.shape[1]))
           for i in range(0, len(leaf), step)]
    return float(np.mean(tot))

def fit_tier(t):
    fs = t['featset']
    if fs == 'text':
        vec = TfidfVectorizer(analyzer='char_wb', lowercase=True, **t['params'])
        Vtr = vec.fit_transform([TEXTS[i] for i in i_tr])
        clf = LogisticRegression(max_iter=3000, C=10).fit(Vtr, y_all[i_tr])
        pv = clf.predict_proba(vec.transform([TEXTS[i] for i in i_va]))[:, 1]
        pt = clf.predict_proba(vec.transform([TEXTS[i] for i in i_te]))[:, 1]
        return (vec, clf), pv, pt, 0.0, None

    cols = CHEAP_COLS if fs == 'cheap' else FULL_COLS
    Xtr, Xva, Xte = F.loc[i_tr, cols], F.loc[i_va, cols], F.loc[i_te, cols]

    if t['kind'] == 'lr':
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), y_all[i_tr])
        return (sc, clf, cols), clf.predict_proba(sc.transform(Xva))[:, 1], \
               clf.predict_proba(sc.transform(Xte))[:, 1], 0.0, None

    if t['kind'] == 'dt':
        clf = DecisionTreeClassifier(random_state=SEED, **t['params']).fit(Xtr, y_all[i_tr])
        nv = float(clf.decision_path(Xte).sum() / len(Xte))
        return (clf, cols), clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1], nv, None

    hist = {}
    clf = lgb.LGBMClassifier(random_state=SEED, n_jobs=1, verbose=-1, **t['params'])
    clf.fit(Xtr, y_all[i_tr], eval_set=[(Xva, y_all[i_va])], eval_metric=['auc', 'binary_logloss'],
            callbacks=[lgb.record_evaluation(hist)])
    key = list(hist.keys())[0]
    hdf = pd.DataFrame({'round': np.arange(1, len(hist[key]['auc']) + 1),
                        'valid_auc': hist[key]['auc'],
                        'valid_logloss': hist[key]['binary_logloss']})
    hdf.to_csv(os.path.join(RUN_DIR, 'history', f"{t['name']}.csv"), index=False)
    nv = node_visits_lgb(clf, Xte.values)
    return (clf, cols), clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1], nv, hdf

HITS = {}          # (tier, fpr_target) -> per-attack correctness vector, for paired tests
BOOT = np.random.RandomState(SEED + 7)

def eval_at_fprs(pv, pt, yva, yte, tier_name=None):
    out = {'auc': float(roc_auc_score(yte, pt))}
    thrs = {}
    for target in FPR_TARGETS:
        thr = float(np.quantile(pv[yva == 0], 1 - target))
        pred = (pt >= thr).astype(int)
        tp = int(((pred == 1) & (yte == 1)).sum()); fp = int(((pred == 1) & (yte == 0)).sum())
        fn = int(((pred == 0) & (yte == 1)).sum()); tn = int(((pred == 0) & (yte == 0)).sum())
        rec = tp / max(tp + fn, 1); prec = tp / max(tp + fp, 1); fpr = fp / max(fp + tn, 1)
        out[f'recall@fpr{target}'] = round(rec, 4)
        out[f'precision@fpr{target}'] = round(prec, 4)
        out[f'fpr_actual@fpr{target}'] = round(fpr, 4)
        out[f'f1@fpr{target}'] = round(2 * prec * rec / max(prec + rec, 1e-9), 4)
        out[f'missed_attacks@fpr{target}'] = fn

        # bootstrap the recall: resample the attacks in the test set with replacement.
        # The threshold is held fixed, because it is a decision rule chosen on validation,
        # not something we want to re-estimate here.
        hits = (pred[yte == 1] == 1).astype(np.uint8)
        bi = BOOT.randint(0, len(hits), size=(N_BOOTSTRAP, len(hits))).astype(np.int32)
        boot_recalls = hits[bi].mean(axis=1)
        lo, hi = np.percentile(boot_recalls, [2.5, 97.5])
        out[f'recall_lo@fpr{target}'] = round(float(lo), 4)
        out[f'recall_hi@fpr{target}'] = round(float(hi), 4)
        out[f'recall_ci_width@fpr{target}'] = round(float(hi - lo), 4)
        if tier_name is not None:
            HITS[(tier_name, target)] = hits

        thrs[target] = thr
    return out, thrs

MODELS, THRESHOLDS = {}, {}
done_df, done = load_done()

for t in TIERS:
    if t['name'] in done:
        print(f"skip {t['name']} (already in results.csv)")
        continue
    t0 = time.time()
    model, pv, pt, nv, hdf = fit_tier(t)
    metrics, thrs = eval_at_fprs(pv, pt, y_all[i_va], y_all[i_te], t['name'])
    MODELS[t['name']] = (t, model)
    THRESHOLDS[t['name']] = thrs
    row = {'tier': t['name'], 'kind': t['kind'], 'featset': t['featset'],
           'params': json.dumps(t['params'], default=str),
           'train_seconds': round(time.time() - t0, 2),
           'node_visits_per_request': round(nv, 1),
           'n_rounds': int(len(hdf)) if hdf is not None else 0, **metrics}
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', index=False,
                               header=not os.path.exists(RESULTS_CSV))
    print(f"{t['name']:24s} AUC {metrics['auc']:.4f}  recall@0.1%FPR "
          f"{metrics['recall@fpr0.001']:.4f} [{metrics['recall_lo@fpr0.001']:.4f}, {metrics['recall_hi@fpr0.001']:.4f}]"
          f"  nodes {nv:8.1f}  ({row['train_seconds']}s)")

# re-fit anything that was skipped, we need the objects in memory for timing
for t in TIERS:
    if t['name'] not in MODELS:
        model, pv, pt, nv, _ = fit_tier(t)
        MODELS[t['name']] = (t, model)
        _, THRESHOLDS[t['name']] = eval_at_fprs(pv, pt, y_all[i_va], y_all[i_te], t['name'])

np.savez(os.path.join(RUN_DIR, 'hit_vectors.npz'),
         **{f'{k[0]}__{k[1]}': v for k, v in HITS.items()})
print('\nresults.csv now has', len(pd.read_csv(RESULTS_CSV)), 'rows')
pd.read_csv(RESULTS_CSV)[['tier','auc','recall@fpr0.001','recall_lo@fpr0.001','recall_hi@fpr0.001',
                          'recall@fpr0.01','node_visits_per_request']]

---
## Cell 10. Single-request latency

This is the measurement the paper turns on, so read what it does.

For each tier we time the **whole serve-time path**: take one raw request, extract its features, score it, produce a decision. One request at a time, single-threaded, because that is the situation an inline filter is actually in. Batch prediction would be much faster per request and would not represent the deployment.

Each request is timed several times and we keep the **fastest** run, which is standard practice for removing scheduler noise. We then report p50 and p99 across requests. Tail latency matters more than the mean here: a filter that is usually fast but occasionally slow still breaks the deadline.

Colab vCPUs are shared, so treat the **ratios between tiers** as the result, not the absolute microseconds. The raw per-request timings are saved so you can report the spread honestly.

Each tier is measured over several full sweeps with the running order reshuffled every time, and the reported number is the median across sweeps. Check the `p50_sweep_spread_pct` column: if it is wider than the gap between two tiers, those tiers are indistinguishable on this hardware and you should say so rather than ranking them.

**Expect a surprise here.** The logistic regression often comes out *slower* than the gradient boosting tiers, and the small and large GBDTs often come out almost identical. That is not a bug. Scoring one row at a time is dominated by fixed per-call overhead in the library, not by the arithmetic of the model. This is exactly why the paper reports node visits as well as latency: wall-clock time alone hides a 40x difference in actual compute. Say this explicitly in the results section, it is one of the more useful things the experiment shows.

In [ ]:
def make_scorer(name):
    t, model = MODELS[name]
    fs = t['featset']
    if fs == 'text':
        vec, clf = model
        def score(r):
            return clf.predict_proba(vec.transform([payload_text(r)]))[0, 1]
        return score
    if t['kind'] == 'lr':
        sc, clf, cols = model
        def score(r):
            f = feats_cheap(r)
            x = np.array([[f[c] for c in cols]])
            return clf.predict_proba(sc.transform(x))[0, 1]
        return score
    if t['kind'] == 'dt':
        clf, cols = model
        def score(r):
            f = feats_cheap(r)
            return clf.predict_proba(np.array([[f[c] for c in cols]]))[0, 1]
        return score
    clf, cols = model
    booster = clf.booster_
    use_full = (fs == 'full')
    def score(r):
        f = feats_cheap(r)
        if use_full:
            f.update(feats_expensive(r))
        return booster.predict(np.array([[f[c] for c in cols]]))[0]
    return score

def time_path(fn, sample, reps):
    out = np.empty(len(sample))
    for k, r in enumerate(sample):
        best = np.inf
        for _ in range(reps):
            t0 = time.perf_counter_ns(); fn(r); t1 = time.perf_counter_ns()
            best = min(best, t1 - t0)
        out[k] = best / 1000.0          # microseconds
    return out

sample_idx = i_te[:TIMING_SAMPLE]
sample = [records[i] for i in sample_idx]

# warm-up so the first tier is not penalised by lazy imports
_w = make_scorer(TIERS[0]['name'])
for r in sample[:50]: _w(r)

# Multiple full sweeps, tier order reshuffled each time. A fixed order lets machine
# noise line up with tier identity, which is how a 5x compute difference can show up
# as a reversed latency ordering. We take the median across sweeps.
sweep_rng = np.random.RandomState(SEED)
PATHS = {t['name']: make_scorer(t['name']) for t in TIERS}
PATHS['X_featurize_cheap_only'] = lambda r: feats_cheap(r)
PATHS['X_featurize_full_only']  = lambda r: (feats_cheap(r), feats_expensive(r))

for fn in PATHS.values():
    for r in sample[:30]: fn(r)          # warm every path before any of them is measured

per_sweep, raw_last = [], {}
for sweep in range(TIMING_SWEEPS):
    order = list(PATHS.keys())
    sweep_rng.shuffle(order)
    for name in order:
        us = time_path(PATHS[name], sample, TIMING_REPEATS)
        raw_last[name] = us
        per_sweep.append({'sweep': sweep, 'tier': name,
                          'p50_us': float(np.percentile(us, 50)),
                          'p90_us': float(np.percentile(us, 90)),
                          'p99_us': float(np.percentile(us, 99)),
                          'mean_us': float(us.mean()), 'max_us': float(us.max()),
                          **{f'meets_{D}us_pct': 100.0 * float((us <= D).mean()) for D in DEADLINES_US}})
    print(f'sweep {sweep+1}/{TIMING_SWEEPS} done')

S = pd.DataFrame(per_sweep)
S.to_csv(os.path.join(RUN_DIR, 'timing_per_sweep.csv'), index=False)
for name, us in raw_last.items():
    np.save(os.path.join(RUN_DIR, 'timing', f'{name}.npy'), us)

agg = S.groupby('tier').median(numeric_only=True).drop(columns=['sweep'])
spread = S.groupby('tier')['p50_us'].agg(['min', 'max'])
agg['p50_sweep_min'] = spread['min']
agg['p50_sweep_max'] = spread['max']
agg['p50_sweep_spread_pct'] = (100 * (spread['max'] - spread['min']) / agg['p50_us']).round(1)
timing_rows = agg.round(1).reset_index().to_dict('records')

print()
for r in sorted(timing_rows, key=lambda x: x['p50_us']):
    print(f"{r['tier']:24s} p50 {r['p50_us']:8.1f}us  p99 {r['p99_us']:9.1f}us  "
          f"sweep spread {r['p50_sweep_spread_pct']:5.1f}%  within 1ms: {r['meets_1000us_pct']:6.2f}%")

T = pd.DataFrame(timing_rows)
T.to_csv(os.path.join(RUN_DIR, 'timing_summary.csv'), index=False)
print('\nsaved timing_summary.csv')
T

---
## Cell 11. Correcting for a realistic attack rate

CSIC is roughly 40% attacks. No storefront looks like that. This cell recomputes precision as if attacks were 1% of traffic.

It does this with the arithmetic of base rates rather than by throwing data away:

    precision = (pi * recall) / (pi * recall + (1 - pi) * FPR)

where `pi` is the assumed attack share. Recall and FPR do not depend on how common attacks are, so they carry straight over from the full test set and you get realistic precision with no extra sampling noise. Subsampling the test set down to 1% attacks would leave only a few dozen attacks and make every number jump around.

Watch the `alarms_per_true_attack` column. It is how many false alarms an analyst wades through for each real attack caught, and at a 1% base rate it is usually the number that decides whether a tier is deployable at all.

In [ ]:
PI = REALISTIC_ATTACK_RATE          # assumed attack share of real storefront traffic
res_now = pd.read_csv(RESULTS_CSV).drop_duplicates('tier', keep='last')

rows = []
for _, r in res_now.iterrows():
    row = {'tier': r['tier'], 'assumed_attack_rate': PI}
    for target in FPR_TARGETS:
        rec = float(r[f'recall@fpr{target}'])
        fpr = float(r[f'fpr_actual@fpr{target}'])
        prec = (PI * rec) / max(PI * rec + (1 - PI) * fpr, 1e-12)
        row[f'recall@fpr{target}'] = round(rec, 4)
        row[f'precision_real@fpr{target}'] = round(prec, 4)
        row[f'false_alarms_per_10k@fpr{target}'] = round(1e4 * (1 - PI) * fpr, 1)
        row[f'alarms_per_true_attack@fpr{target}'] = round((1 - prec) / max(prec, 1e-12), 2)
    rows.append(row)

R = pd.DataFrame(rows)
R.to_csv(os.path.join(RUN_DIR, 'realistic_base_rate.csv'), index=False)
print(f'precision recomputed at a {PI:.1%} attack prior, using the full test set')
print('saved realistic_base_rate.csv')
R

---
## Cell 12. Join everything and compute the budget table

Merges detection quality, compute cost and latency into the single table the paper is built around, then derives the two summary columns:

- **attacks caught per 1000 node visits**, a hardware-independent efficiency measure
- **deadline-feasible recall**, which is a tier's recall if and only if it meets the deadline at p99, and zero otherwise

The second column is the paper's argument in one number: a detector that misses the deadline does not get credit for its accuracy.

The cell then runs a **paired bootstrap** over every pair of tiers. Each pair is compared on the same resampled attacks, so the interval is on the difference itself rather than two separate intervals you have to eyeball for overlap. If an interval spans zero, those two tiers caught statistically indistinguishable numbers of attacks and you must not rank them in the paper. Results go to `paired_comparisons.csv`, and the top two tiers at the strict operating point are printed.

In [ ]:
res = pd.read_csv(RESULTS_CSV).drop_duplicates('tier', keep='last')
tim = pd.read_csv(os.path.join(RUN_DIR, 'timing_summary.csv'))
real = pd.read_csv(os.path.join(RUN_DIR, 'realistic_base_rate.csv'))

real_cols = [c for c in real.columns if c == 'tier' or c.startswith(('precision_real','false_alarms','alarms_per'))]
M = res.merge(tim, on='tier', how='left').merge(real[real_cols], on='tier', how='left')
M['attacks_caught_per_1k_nodes'] = (1000 * M['recall@fpr0.01'] /
                                    M['node_visits_per_request'].replace(0, np.nan)).round(3)
for D in DEADLINES_US:
    M[f'feasible_recall@{D}us'] = np.where(M['p99_us'] <= D, M['recall@fpr0.01'], 0.0)

M.to_csv(os.path.join(RUN_DIR, 'budget_table.csv'), index=False)

manifest = {'run_kind': RUN_KIND, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'seed': SEED, 'smoke': SMOKE, 'n_requests_used': int(len(idx)),
            'timing_sample': TIMING_SAMPLE, 'timing_repeats': TIMING_REPEATS,
            'fpr_targets': FPR_TARGETS, 'deadlines_us': DEADLINES_US,
            'realistic_attack_rate': REALISTIC_ATTACK_RATE,
            'lightgbm': lgb.__version__, 'sklearn': sklearn.__version__,
            'numpy': np.__version__, 'cpu_count': os.cpu_count(),
            'dataset': 'CSIC 2010 (36000 normal + 25065 anomalous)'}
with open(os.path.join(RUN_DIR, 'manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

# ---- paired bootstrap: is one tier really better than another, or is it noise? ----
# The same resampled attacks are used for every tier, so the comparison is paired and
# the interval reflects the difference directly rather than two separate intervals.
hv = np.load(os.path.join(RUN_DIR, 'hit_vectors.npz'))
pair_rows = []
for target in FPR_TARGETS:
    names = [t['name'] for t in TIERS if f"{t['name']}__{target}" in hv]
    if len(names) < 2: continue
    H = np.vstack([hv[f'{n}__{target}'] for n in names]).astype(np.uint8)
    rs = np.random.RandomState(SEED + 11)
    bi = rs.randint(0, H.shape[1], size=(N_BOOTSTRAP, H.shape[1])).astype(np.int32)
    boot = H[:, bi].mean(axis=2)                     # (n_tiers, N_BOOTSTRAP)
    for a in range(len(names)):
        for b in range(len(names)):
            if a >= b: continue
            d = boot[a] - boot[b]
            lo, hi = np.percentile(d, [2.5, 97.5])
            pair_rows.append({'fpr_target': target, 'tier_a': names[a], 'tier_b': names[b],
                              'recall_diff': round(float(H[a].mean() - H[b].mean()), 4),
                              'diff_lo': round(float(lo), 4), 'diff_hi': round(float(hi), 4),
                              'significant': bool(lo > 0 or hi < 0)})

PAIRS = pd.DataFrame(pair_rows)
PAIRS.to_csv(os.path.join(RUN_DIR, 'paired_comparisons.csv'), index=False)

strict = min(FPR_TARGETS)
top = M[M.tier.str.startswith('T')].nlargest(2, f'recall@fpr{strict}')['tier'].tolist()
if len(top) == 2:
    q = PAIRS[(PAIRS.fpr_target == strict) &
              (((PAIRS.tier_a == top[0]) & (PAIRS.tier_b == top[1])) |
               ((PAIRS.tier_a == top[1]) & (PAIRS.tier_b == top[0])))]
    if len(q):
        r = q.iloc[0]
        verdict = 'SIGNIFICANT' if r['significant'] else 'NOT significant (interval spans zero)'
        print(f"Top two at {strict:.1%} FPR: {r['tier_a']} vs {r['tier_b']}")
        print(f"  recall difference {r['recall_diff']:+.4f}  95% CI [{r['diff_lo']:+.4f}, {r['diff_hi']:+.4f}]  -> {verdict}\n")

print('saved budget_table.csv, manifest.json and paired_comparisons.csv\n')
show = ['tier','auc','recall@fpr0.01','precision_real@fpr0.01','node_visits_per_request',
        'p50_us','p99_us','attacks_caught_per_1k_nodes','feasible_recall@1000us']
M[show]

---
## Cell 13. Figures

Three plots, saved as 300 dpi PNGs ready to drop into the paper:

1. Recall against compute cost, the Pareto view
2. Recall against p99 latency, with deadline lines drawn in
3. Validation AUC per boosting round, showing where the large models stop earning their size

In [ ]:
import matplotlib.pyplot as plt
FIG = os.path.join(RUN_DIR, 'figures')
P = M[M.tier.str.startswith('T')].copy()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(P['node_visits_per_request'].clip(lower=1), P['recall@fpr0.01'], s=70, zorder=3)
for _, r in P.iterrows():
    ax.annotate(r['tier'].replace('_', ' '), (max(r['node_visits_per_request'], 1), r['recall@fpr0.01']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xscale('log'); ax.set_xlabel('node visits per request (log scale)')
ax.set_ylabel('recall at 1% FPR'); ax.set_title('Detection vs compute cost')
ax.grid(alpha=.3, zorder=0); fig.tight_layout()
fig.savefig(os.path.join(FIG, 'fig1_recall_vs_nodes.png'), dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(P['p99_us'], P['recall@fpr0.01'], s=70, zorder=3)
for _, r in P.iterrows():
    ax.annotate(r['tier'].replace('_', ' '), (r['p99_us'], r['recall@fpr0.01']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
for D in [250, 1000]:
    ax.axvline(D, ls='--', c='crimson', alpha=.6)
    ax.text(D, ax.get_ylim()[0], f' {D}us deadline', rotation=90, va='bottom', fontsize=8, c='crimson')
ax.set_xscale('log'); ax.set_xlabel('p99 single-request latency, us (log scale)')
ax.set_ylabel('recall at 1% FPR'); ax.set_title('Detection vs tail latency under deadlines')
ax.grid(alpha=.3, zorder=0); fig.tight_layout()
fig.savefig(os.path.join(FIG, 'fig2_recall_vs_p99.png'), dpi=300); plt.show()

hdir = os.path.join(RUN_DIR, 'history')
hfiles = sorted(os.listdir(hdir)) if os.path.isdir(hdir) else []
if hfiles:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for fn in hfiles:
        h = pd.read_csv(os.path.join(hdir, fn))
        ax.plot(h['round'], h['valid_auc'], label=fn.replace('.csv', '').replace('_', ' '))
    ax.set_xlabel('boosting round'); ax.set_ylabel('validation AUC')
    ax.set_title('Where extra trees stop paying'); ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, 'fig3_auc_per_round.png'), dpi=300); plt.show()

print('figures saved to', FIG)

---
## Cell 14. Run summary

Prints the sentences you will actually write the paper around, and lists every file saved to Drive.

In [ ]:
P = M[M.tier.str.startswith('T')].copy()
best_acc = P.loc[P['recall@fpr0.01'].idxmax()]
best_eff = P.loc[P['attacks_caught_per_1k_nodes'].idxmax()]

print('=' * 72)
print(f'RUN: {RUN_KIND.upper()}   requests used: {len(idx):,}   {manifest["timestamp"]}')
print('=' * 72)
print(f"\nMost accurate tier      : {best_acc['tier']}")
print(f"  recall@1%FPR {best_acc['recall@fpr0.01']:.4f} | nodes {best_acc['node_visits_per_request']:.0f} | p99 {best_acc['p99_us']:.0f}us")
print(f"\nMost cost-efficient tier: {best_eff['tier']}")
print(f"  recall@1%FPR {best_eff['recall@fpr0.01']:.4f} | nodes {best_eff['node_visits_per_request']:.0f} | p99 {best_eff['p99_us']:.0f}us")

if best_acc['tier'] != best_eff['tier']:
    dn = best_acc['node_visits_per_request'] / max(best_eff['node_visits_per_request'], 1)
    dr = best_acc['recall@fpr0.01'] - best_eff['recall@fpr0.01']
    print(f"\n>> The most accurate tier costs {dn:.1f}x the compute for {dr*100:.1f} more points of recall.")
    print('   That gap is the paper.')
else:
    print('\n>> Accuracy and efficiency picked the same tier. Widen the tier range in Cell 8.')

fo = M[M.tier == 'X_featurize_cheap_only']
if len(fo):
    t3 = P[P.tier.str.contains('T3')]
    if len(t3):
        share = 100 * fo.iloc[0]['p50_us'] / max(t3.iloc[0]['p50_us'], 1e-9)
        print(f"\n>> Feature extraction is {share:.0f}% of the smallest tier's total latency.")

print('\nFiles written to', RUN_DIR)
for dirpath, _, fnames in os.walk(RUN_DIR):
    for fn in sorted(fnames):
        p = os.path.join(dirpath, fn)
        print(f"  {os.path.relpath(p, RUN_DIR):45s} {os.path.getsize(p)/1024:8.1f} KB")

if SMOKE:
    print('\n' + '!' * 72)
    print('This was the SMOKE TEST. Numbers are not paper-ready.')
    print('Next: set SMOKE = False in Cell 2, then Runtime > Restart session, then Run all.')
    print('!' * 72)